# 10 · HMDA：特征块消融（Colab GPU，五折）

数据检查（2026-10-14）没有发现数据错误：训练/测试对抗验证 AUC 0.501、无冲突重复、ID 无泄漏。因此这里不「清洗」，而是检验**给模型加入它不容易自己算出的信息**能否提高五折 AUC。

|特征块|新增列|想法|
|---|---|---|
|`dti`|`dti_lower`、`dti_upper`|DTI 类别的数值上下界（与 04 相同）|
|`ltv`|`ltv_calc`、`ltv_gap`|由贷款额/房价算出的 LTV，以及与申报 LTV 的差|
|`income`|`income_to_area`、`loan_to_income`、`value_to_income`|收入相对地区收入、贷款额和房价相对收入|
|`missing`|`missing_pattern`（类别）、`missing_count`|关键字段的缺失组合；LTV 缺失时批准率 54%，有值时 89%|
|`freq`|`freq_county_code`、`freq_derived_msa-md`、`freq_state_code`|地区在训练+测试中出现的次数（不使用标签）|

**协议**：固定 seed=42 五折；每个变体 = 原始 56 列 + 一个特征块，`F6_all` 为全部块。模型统一用快速配置（CatBoost GPU，depth 8、lr 0.05、L2=3、Bernoulli 抽样 0.8、固定 1600 轮，每折约 2 分钟），只比较**同折相对 `F0_base` 的差**。比值的分母必须 > 0，否则为缺失，不加任意 epsilon。

**判读**：五折平均提升 ≥ 0.0003 且至少 4/5 折同向的块才保留；保留的块再合并，换成 09 选出的参数做最终训练。

**操作**：Colab 选 **T4 GPU**，从上到下运行。约 7 个变体 × 5 折 × 2 分钟 ≈ 70–80 分钟；每个（变体, 折）完成即存到 Drive `artifacts/feature_ablation/<EXPERIMENT_ID>/`，断线后从头运行可续跑。可以和 09 同时在另一个 Colab 窗口运行。

## 1. 环境、GPU 与配置

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')

PROJECT_DIR = Path(os.environ.get(
    'HMDA_PROJECT_DIR',
    '/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis',
)).expanduser().resolve()
PREPARED_DIR = PROJECT_DIR / 'artifacts' / 'prepared'
SEED = 42
CATBOOST_VERSION = '1.2.10'
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'找不到项目目录：{PROJECT_DIR}；请修改 PROJECT_DIR')

try:
    installed_catboost = importlib.metadata.version('catboost')
except importlib.metadata.PackageNotFoundError:
    installed_catboost = None
if installed_catboost != CATBOOST_VERSION:
    if 'catboost' in sys.modules:
        raise RuntimeError('当前运行时已经导入另一版 CatBoost；请重启 Colab 运行时后从头运行')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        f'catboost=={CATBOOST_VERSION}',
    ])
    importlib.invalidate_caches()

import catboost
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

if catboost.__version__ != CATBOOST_VERSION:
    raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}；实际为 {catboost.__version__}')

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('未找到 nvidia-smi：请在 Colab 更改运行时类型，选择 T4 GPU 后从头运行')
gpu_probe = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    text=True, capture_output=True, check=False,
)
if gpu_probe.returncode != 0 or not gpu_probe.stdout.strip():
    raise RuntimeError('NVIDIA GPU 不可用：请在 Colab 选择 T4 GPU 后从头运行')
GPU_NAME = gpu_probe.stdout.strip().splitlines()[0]

EXPERIMENT_ID = 'feature_blocks_v1'
FOLDS = [1, 2, 3, 4, 5]
RUN_ONLY = None             # 例如 ['F0_base', 'F2_ltv']；None 表示全部变体
OUTPUT_DIR = PROJECT_DIR / 'artifacts' / 'feature_ablation' / EXPERIMENT_ID

FAST_PARAMS = dict(
    iterations=1600, depth=8, learning_rate=0.05, l2_leaf_reg=3,
    bootstrap_type='Bernoulli', subsample=0.8,
    loss_function='Logloss', nan_mode='Min', random_seed=SEED,
    task_type='GPU', devices='0', thread_count=4,
    use_best_model=False, allow_writing_files=False, verbose=False,
)
VARIANTS = {
    'F0_base': [],
    'F1_dti': ['dti'],
    'F2_ltv': ['ltv'],
    'F3_income': ['income'],
    'F4_missing': ['missing'],
    'F5_freq': ['freq'],
    'F6_all': ['dti', 'ltv', 'income', 'missing', 'freq'],
}
if RUN_ONLY is not None:
    unknown = set(RUN_ONLY) - set(VARIANTS)
    if unknown:
        raise ValueError(f'RUN_ONLY 中有未知变体：{sorted(unknown)}')
print(f'项目：{PROJECT_DIR}')
print(f'GPU：{GPU_NAME}；CatBoost {catboost.__version__}；折：{FOLDS}；输出：{OUTPUT_DIR}')

Mounted at /content/drive
项目：/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis
GPU：Tesla T4, 15360 MiB；CatBoost 1.2.10；折：[1, 2, 3, 4, 5]；输出：/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis/artifacts/feature_ablation/feature_blocks_v1


## 2. 回读并核对 01 的数据契约（与 04 相同）

In [2]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


manifest_path = PREPARED_DIR / 'manifest.json'
schema_path = PREPARED_DIR / 'schema.json'
if not manifest_path.is_file() or not schema_path.is_file():
    raise FileNotFoundError(
        f'缺少 {manifest_path} 或 {schema_path}；请先运行 01_hmda_data_preparation.ipynb'
    )
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
schema = json.loads(schema_path.read_text(encoding='utf-8'))
require(manifest['schema_version'] == schema['schema_version'] == '1.0', '数据契约版本不匹配')
require(manifest['seed'] == SEED and manifest['n_splits'] == 5, '五折配置不匹配')
require(set(manifest['source_files']) == {'train.csv', 'test.csv', 'sample_submission.csv'},
        '原始文件清单不完整')
expected_prepared = {
    'train_features.csv', 'test_features.csv', 'train_labels.csv',
    'test_ids.csv', 'folds.csv', 'audit.csv', 'schema.json',
}
require(set(manifest['output_files']) == expected_prepared, '准备文件清单不完整')
for name, info in manifest['source_files'].items():
    path = PROJECT_DIR / name
    require(path.is_file(), f'缺少原始数据：{path}')
    require(sha256_file(path) == info['sha256'], f'原始文件已变化：{name}；请先重跑 01')
for name, info in manifest['output_files'].items():
    path = PREPARED_DIR / name
    require(path.is_file(), f'缺少准备文件：{path}')
    require(sha256_file(path) == info['sha256'], f'准备文件哈希不匹配：{name}；请先重跑 01')

FEATURE_COLUMNS = schema['feature_columns']
NUMERIC_COLUMNS = schema['numeric_columns']
CATEGORICAL_COLUMNS = schema['categorical_columns']
require(len(FEATURE_COLUMNS) == 56 and len(NUMERIC_COLUMNS) == 13
        and len(CATEGORICAL_COLUMNS) == 43, '预期为 56 列：13 数值、43 类别')
require(set(FEATURE_COLUMNS) == set(NUMERIC_COLUMNS + CATEGORICAL_COLUMNS),
        '数值/类别列分组不完整')
require(not {'id', 'accepted'} & set(FEATURE_COLUMNS), 'ID 或标签不能作为特征')


def read_features(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    require(frame.columns.tolist() == FEATURE_COLUMNS, f'特征列或顺序不匹配：{path.name}')
    for column in NUMERIC_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column].replace('', np.nan), errors='raise',
        ).astype('float64')
    require(not np.isinf(frame[NUMERIC_COLUMNS].to_numpy()).any(), '数值列含无穷大')
    require(not frame[CATEGORICAL_COLUMNS].eq('').any().any(), '类别列含空字符串')
    return frame


X_base = read_features(PREPARED_DIR / 'train_features.csv')
X_test_base = read_features(PREPARED_DIR / 'test_features.csv')
labels = pd.read_csv(PREPARED_DIR / 'train_labels.csv', dtype=str, keep_default_na=False)
test_id_frame = pd.read_csv(PREPARED_DIR / 'test_ids.csv', dtype=str, keep_default_na=False)
fold_frame = pd.read_csv(PREPARED_DIR / 'folds.csv', dtype=str, keep_default_na=False)
sample_submission = pd.read_csv(PROJECT_DIR / 'sample_submission.csv', dtype=str, keep_default_na=False)

require(labels.columns.tolist() == ['id', 'accepted'], '训练标签列不匹配')
require(test_id_frame.columns.tolist() == ['id'], '测试 ID 列不匹配')
require(fold_frame.columns.tolist() == ['id', 'fold'], '折号文件列不匹配')
require(sample_submission.columns.tolist() == ['id', 'accepted'], '提交模板列不匹配')
require(len(X_base) == len(labels) == len(fold_frame) == 105000, '训练行数不匹配')
require(len(X_test_base) == len(test_id_frame) == len(sample_submission) == 45000,
        '测试行数不匹配')

train_ids = labels['id']
test_ids = test_id_frame['id']
require(train_ids.is_unique and test_ids.is_unique, 'ID 重复')
require(train_ids.ne('').all() and test_ids.ne('').all(), 'ID 为空')
require(train_ids.equals(fold_frame['id']), '折号 ID 与训练行错位')
require(test_ids.equals(sample_submission['id']), '测试 ID 与提交模板顺序不符')
require(set(train_ids).isdisjoint(test_ids), '训练与测试 ID 重叠')
require(set(labels['accepted']) == {'0', '1'}, 'accepted 需同时包含 0 和 1')
raw_train_identity = pd.read_csv(
    PROJECT_DIR / 'train.csv', usecols=['id', 'accepted'],
    dtype=str, keep_default_na=False,
)
raw_test_identity = pd.read_csv(
    PROJECT_DIR / 'test.csv', usecols=['id'], dtype=str, keep_default_na=False,
)
require(train_ids.equals(raw_train_identity['id'])
        and labels['accepted'].equals(raw_train_identity['accepted']),
        '准备标签与原始 train.csv 的 ID/标签不一致')
require(test_ids.equals(raw_test_identity['id']),
        '准备测试 ID 与原始 test.csv 不一致')
y = labels['accepted'].astype('int8')
require(set(fold_frame['fold']) == {'1', '2', '3', '4', '5'}, '折号应为 1..5')
fold_assignment = fold_frame['fold'].astype('int8').to_numpy()

expected_folds = np.zeros(len(X_base), dtype=np.int8)
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (_, valid_index) in enumerate(splitter.split(X_base, y), start=1):
    expected_folds[valid_index] = fold
require(np.array_equal(fold_assignment, expected_folds), '折号并非固定 seed=42 五折')
DATA_HASHES = {
    'manifest.json': sha256_file(manifest_path),
    **{name: info['sha256'] for name, info in manifest['output_files'].items()},
}
print(f'数据核对通过：训练 {len(X_base):,}，测试 {len(X_test_base):,}，56 个原始特征。')
display(pd.crosstab(fold_assignment, y, rownames=['fold'], colnames=['accepted']))

def checked_probabilities(model, features):
    require(list(model.classes_) == [0, 1], '类别顺序不是 [0, 1]')
    probability = np.asarray(model.predict_proba(features)[:, 1], dtype=float)
    require(probability.ndim == 1 and len(probability) == len(features)
            and np.isfinite(probability).all()
            and ((probability >= 0) & (probability <= 1)).all(),
            '预测概率无效')
    return probability


def write_json(path, payload):
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def fixed_config(output_dir, config):
    """首次运行写入配置；之后配置必须完全相同，避免把不同实验的结果混在一个目录。"""
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / 'config.json'
    if path.is_file():
        saved = json.loads(path.read_text(encoding='utf-8'))
        require(saved == config,
                f'{output_dir.name} 已有不同配置的结果；改动配置时请改 EXPERIMENT_ID')
    else:
        write_json(path, config)

数据核对通过：训练 105,000，测试 45,000，56 个原始特征。


accepted,0,1
fold,,
1,3151,17849
2,3151,17849
3,3151,17849
4,3151,17849
5,3152,17848


## 3. DTI 上下界（与 04 相同）

In [3]:
DTI_BOUNDS = {
    **{str(value): (float(value), float(value)) for value in range(36, 50)},
    '20%-<30%': (20.0, 30.0),
    '30%-<36%': (30.0, 36.0),
    '50%-60%': (50.0, 60.0),
    '<20%': (np.nan, 20.0),
    '>60%': (60.0, np.nan),
    'MISSING': (np.nan, np.nan),
}


def add_dti_bounds(frame):
    unknown = set(frame['debt_to_income_ratio'].unique()) - set(DTI_BOUNDS)
    require(not unknown, f'DTI 出现未知类别：{sorted(map(str, unknown))}')
    result = frame.copy()
    pairs = frame['debt_to_income_ratio'].map(DTI_BOUNDS)
    result['dti_lower'] = pairs.map(lambda pair: pair[0]).astype('float64')
    result['dti_upper'] = pairs.map(lambda pair: pair[1]).astype('float64')
    require(result.columns.tolist() == FEATURE_COLUMNS + ['dti_lower', 'dti_upper'],
            'DTI 特征列或顺序错误')
    require(result['debt_to_income_ratio'].equals(frame['debt_to_income_ratio']),
            '原始 DTI 类别被修改')
    return result


X_dti = add_dti_bounds(X_base)
X_test_dti = add_dti_bounds(X_test_base)
print(f'候选特征：原始 56 列 + 2 列 DTI 上下界 = {X_dti.shape[1]} 列。')
display(X_dti[['debt_to_income_ratio', 'dti_lower', 'dti_upper']].head())

候选特征：原始 56 列 + 2 列 DTI 上下界 = 58 列。


,debt_to_income_ratio,dti_lower,dti_upper
0,20%-<30%,20.0,30.0
1,49,49.0,49.0
2,20%-<30%,20.0,30.0
3,44,44.0,44.0
4,<20%,NaN,20.0


## 4. 构建特征块

训练与测试一起构建，保证口径一致；只有 `freq` 用到测试集的分布（只计数，不涉及标签）。原 56 列保持不变，新列追加在后面。

In [4]:
def safe_ratio(numerator, denominator):
    valid = numerator.notna() & denominator.notna() & (denominator > 0)
    return (numerator / denominator.where(valid)).where(valid).astype('float64')


def build_blocks(train, test):
    both = pd.concat([train, test], ignore_index=True)
    blocks = {}
    blocks['dti'] = add_dti_bounds(both)[['dti_lower', 'dti_upper']]

    ltv_calc = 100 * safe_ratio(both['loan_amount'], both['property_value'])
    blocks['ltv'] = pd.DataFrame({
        'ltv_calc': ltv_calc,
        'ltv_gap': both['loan_to_value_ratio'] - ltv_calc,
    })

    income_dollars = both['income'] * 1000      # HMDA 收入单位为千美元
    blocks['income'] = pd.DataFrame({
        'income_to_area': safe_ratio(income_dollars, both['ffiec_msa_md_median_family_income']),
        'loan_to_income': safe_ratio(both['loan_amount'], income_dollars),
        'value_to_income': safe_ratio(both['property_value'], income_dollars),
    })

    missing = pd.DataFrame({
        'ltv': both['loan_to_value_ratio'].isna(),
        'pv': both['property_value'].isna(),
        'inc': both['income'].isna(),
        'term': both['loan_term'].isna(),
        'dti': both['debt_to_income_ratio'].eq('MISSING'),
        'age': both['applicant_age'].eq('MISSING'),
    })
    blocks['missing'] = pd.DataFrame({
        'missing_pattern': 'm' + missing.astype(int).astype(str).agg(''.join, axis=1),
        'missing_count': missing.sum(axis=1).astype('float64'),
    })

    freq = {}
    for column in ['county_code', 'derived_msa-md', 'state_code']:
        counts = both[column].map(both[column].value_counts()).astype('float64')
        freq[f'freq_{column}'] = counts.where(both[column].ne('MISSING'))
    blocks['freq'] = pd.DataFrame(freq)

    n_train = len(train)
    out = {}
    for name, frame in blocks.items():
        require(len(frame) == len(both), f'{name} 行数不符')
        out[name] = (frame.iloc[:n_train].reset_index(drop=True),
                     frame.iloc[n_train:].reset_index(drop=True))
    return out


BLOCKS = build_blocks(X_base, X_test_base)
NEW_CATEGORICAL = {'missing_pattern'}


def variant_frames(blocks):
    train_parts = [X_base] + [BLOCKS[b][0] for b in blocks]
    test_parts = [X_test_base] + [BLOCKS[b][1] for b in blocks]
    train = pd.concat(train_parts, axis=1)
    test = pd.concat(test_parts, axis=1)
    require(train.columns.is_unique and train.columns.equals(test.columns), '变体列不一致')
    require(train.iloc[:, :len(FEATURE_COLUMNS)].equals(X_base), '原始 56 列被修改')
    cats = CATEGORICAL_COLUMNS + [c for c in train.columns if c in NEW_CATEGORICAL]
    return train, test, cats


for name, (train_block, _) in BLOCKS.items():
    described = train_block.describe(include='all').T
    print(f'\n[{name}] 缺失率：', train_block.isna().mean().round(3).to_dict())
    display(described.iloc[:, :6])


[dti] 缺失率： {'dti_lower': 0.117, 'dti_upper': 0.117}


,count,mean,std,min,25%,50%
dti_lower,92671.0,39.680947,10.654997,20.0,30.0,42.0
dti_upper,92730.0,41.033290,9.699260,20.0,36.0,40.0



[ltv] 缺失率： {'ltv_calc': 0.045, 'ltv_gap': 0.118}


,count,mean,std,min,25%,50%
ltv_calc,100244.0,82.070292,54.826068,0.302115,74.683544,88.235294
ltv_gap,92643.0,5.568211,439.329812,-7717.790000,-0.719136,0.000000



[income] 缺失率： {'income_to_area': 0.054, 'loan_to_income': 0.053, 'value_to_income': 0.088}


,count,mean,std,min,25%,50%
income_to_area,99343.0,1.563771,3.274112,-6.550349,0.746269,1.108830
loan_to_income,99468.0,2.891506,4.565102,0.002606,1.810560,2.692308
value_to_income,95792.0,3.713296,6.426108,0.009722,2.378641,3.288576



[missing] 缺失率： {'missing_pattern': 0.0, 'missing_count': 0.0}


,count,unique,top,freq,mean,std
missing_pattern,105000,42,m000000,88200,NaN,NaN
missing_count,105000.0,NaN,NaN,NaN,0.336019,0.956439



[freq] 缺失率： {'freq_county_code': 0.013, 'freq_derived_msa-md': 0.0, 'freq_state_code': 0.008}


,count,mean,std,min,25%,50%
freq_county_code,103661.0,364.801787,470.338073,1.0,67.0,200.0
freq_derived_msa-md,105000.0,4000.393038,7326.553843,2.0,353.0,927.0
freq_state_code,104176.0,6209.485976,4859.097575,2.0,2667.0,4441.0


## 5. 五折训练（可续跑）

每个（变体, 折）结果写为 `<变体>__fold<k>.json` 和验证概率 `.csv`。修改已有变体的定义时必须改名或改 `EXPERIMENT_ID`。

In [5]:
fixed_config(OUTPUT_DIR, {
    'purpose': 'feature block ablation, fast GPU CatBoost, paired vs F0_base',
    'seed': SEED,
    'fast_params': {k: v for k, v in FAST_PARAMS.items() if k != 'thread_count'},
    'data_hashes': DATA_HASHES,
})

selected = {n: b for n, b in VARIANTS.items() if RUN_ONLY is None or n in RUN_ONLY}
for fold in FOLDS:
    fit_idx = np.flatnonzero(fold_assignment != fold)
    valid_idx = np.flatnonzero(fold_assignment == fold)
    for name, blocks in selected.items():
        result_path = OUTPUT_DIR / f'{name}__fold{fold}.json'
        if result_path.is_file():
            saved = json.loads(result_path.read_text(encoding='utf-8'))
            require(saved['blocks'] == blocks, f'{name} 的定义已改变；请改变体名')
            print(f'{name} fold {fold}: 已完成 AUC={saved["valid_auc"]:.6f}，跳过')
            continue
        train, _, cats = variant_frames(blocks)
        started = time.perf_counter()
        model = catboost.CatBoostClassifier(**FAST_PARAMS)
        model.fit(train.iloc[fit_idx], y.iloc[fit_idx], cat_features=cats)
        require(int(model.tree_count_) == FAST_PARAMS['iterations'], '轮数不符')
        probability = checked_probabilities(model, train.iloc[valid_idx])
        actual = y.iloc[valid_idx]
        pd.DataFrame({'id': train_ids.iloc[valid_idx].to_numpy(), 'accepted': actual.to_numpy(),
                      'prob': probability}).to_csv(
            OUTPUT_DIR / f'{name}__fold{fold}.csv', index=False, float_format='%.12f')
        result = {
            'variant': name, 'blocks': blocks, 'fold': fold,
            'feature_count': train.shape[1],
            'valid_auc': roc_auc_score(actual, probability),
            'valid_logloss': log_loss(actual, probability, labels=[0, 1]),
            'seconds': time.perf_counter() - started, 'gpu_name': GPU_NAME,
            'finished_at_utc': datetime.now(timezone.utc).isoformat(),
        }
        write_json(result_path, result)
        print(f'{name} fold {fold}: AUC={result["valid_auc"]:.6f}  '
              f'{train.shape[1]} 列  {result["seconds"]/60:.1f} 分钟', flush=True)
        del model, train
        gc.collect()

F0_base fold 1: AUC=0.910164  56 列  2.0 分钟
F1_dti fold 1: AUC=0.910509  58 列  1.8 分钟
F2_ltv fold 1: AUC=0.911949  58 列  1.8 分钟
F3_income fold 1: AUC=0.910457  59 列  1.8 分钟
F4_missing fold 1: AUC=0.910242  58 列  1.8 分钟
F5_freq fold 1: AUC=0.911074  59 列  1.8 分钟
F6_all fold 1: AUC=0.913876  68 列  1.9 分钟
F0_base fold 2: AUC=0.924544  56 列  1.8 分钟
F1_dti fold 2: AUC=0.924560  58 列  1.8 分钟
F2_ltv fold 2: AUC=0.924552  58 列  1.8 分钟
F3_income fold 2: AUC=0.923979  59 列  1.8 分钟
F4_missing fold 2: AUC=0.924329  58 列  1.8 分钟
F5_freq fold 2: AUC=0.924351  59 列  1.8 分钟
F6_all fold 2: AUC=0.926517  68 列  1.9 分钟
F0_base fold 3: AUC=0.916795  56 列  1.8 分钟
F1_dti fold 3: AUC=0.917091  58 列  1.8 分钟
F2_ltv fold 3: AUC=0.918461  58 列  1.8 分钟
F3_income fold 3: AUC=0.917214  59 列  1.8 分钟
F4_missing fold 3: AUC=0.917125  58 列  1.8 分钟
F5_freq fold 3: AUC=0.916382  59 列  1.8 分钟
F6_all fold 3: AUC=0.919224  68 列  1.9 分钟
F0_base fold 4: AUC=0.910882  56 列  1.8 分钟
F1_dti fold 4: AUC=0.911059  58 列  1.8 分钟
F2_ltv

## 6. 与 F0_base 配对比较

`keep` 为建议：五折都完成、平均提升 ≥ 0.0003 且至少 4/5 折优于 F0。

In [6]:
results = pd.DataFrame([json.loads(p.read_text(encoding='utf-8'))
                        for p in sorted(OUTPUT_DIR.glob('*__fold*.json'))])
require(not results.empty, '还没有完成的结果')
auc = results.pivot(index='fold', columns='variant', values='valid_auc')
require('F0_base' in auc.columns, '缺少基线 F0_base')
delta = auc.sub(auc['F0_base'], axis=0)
table = pd.DataFrame({
    'folds_done': auc.notna().sum(),
    'mean_auc': auc.mean(),
    'mean_delta_vs_F0': delta.mean(),
    'folds_better_than_F0': (delta > 0).sum(),
    'mean_logloss': results.groupby('variant')['valid_logloss'].mean(),
    'feature_count': results.groupby('variant')['feature_count'].first(),
})
table['keep'] = ((table['folds_done'] == 5) & (table['mean_delta_vs_F0'] >= 0.0003)
                 & (table['folds_better_than_F0'] >= 4))
table.loc['F0_base', 'keep'] = False
table = table.sort_values('mean_delta_vs_F0', ascending=False)
table.to_csv(OUTPUT_DIR / 'summary.csv')
display(table.round(6))
display(delta.round(6))

,folds_done,mean_auc,mean_delta_vs_F0,folds_better_than_F0,mean_logloss,feature_count,keep
variant,,,,,,,
F6_all,5,0.918660,0.002996,5,0.194817,68,True
F2_ltv,5,0.917752,0.002088,5,0.196240,58,True
F4_missing,5,0.915884,0.000220,4,0.198056,58,False
F3_income,5,0.915767,0.000103,4,0.198426,59,False
F1_dti,5,0.915759,0.000095,4,0.198497,58,False
F0_base,5,0.915664,0.000000,0,0.198734,56,False
F5_freq,5,0.915529,-0.000134,1,0.198844,59,False


variant,F0_base,F1_dti,F2_ltv,F3_income,F4_missing,F5_freq,F6_all
fold,,,,,,,
1,0.0,0.000345,0.001785,0.000293,0.000078,0.000910,0.003712
2,0.0,0.000016,0.000008,-0.000565,-0.000215,-0.000193,0.001973
3,0.0,0.000296,0.001667,0.000419,0.000330,-0.000413,0.002430
4,0.0,0.000176,0.004045,0.000316,0.000182,-0.000638,0.003677
5,0.0,-0.000360,0.002938,0.000054,0.000727,-0.000339,0.003189
